In [1]:
!pip -q install -U transformers peft bitsandbytes accelerate

import os
import zipfile
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.0 MB/s eta 0:00:0000:01


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/kaggle/input/datasets/arnavx/mini-instructgpt-7b-dpo-adapter/kaggle/working/dpo_7b_lora/final"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
    dtype=torch.float16
)

model = PeftModel.from_pretrained(base, adapter_path)
model.eval()
print("Model + adapter loaded ✅")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model + adapter loaded ✅


In [3]:
def ask(q, max_new_tokens=120):
    prompt = f"Human: {q}\n\nAssistant:"
    x = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        y = model.generate(
            **x,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [4]:
import time
print("Model device:", next(model.parameters()).device)

tests = [
    "What is gravity?",
    "How do I make tea?",
    "What is the capital of France?"
]

for t in tests:
    t0 = time.time()
    ans = ask(t, max_new_tokens=80)
    print(f"\nQ: {t}\nA: {ans}\n({time.time()-t0:.1f}s)")

Model device: cuda:0

Q: What is gravity?
A: Gravity is a fundamental force of nature that causes objects with mass to be attracted to each other. It is one of the four known forces in the universe, along with electromagnetism, the strong nuclear force, and the weak nuclear force.

Gravity is responsible for the formation and structure of planets, stars, and galaxies. It keeps our feet on the ground and allows us to jump and fall back down
(10.0s)

Q: How do I make tea?
A: To make a basic cup of tea, follow these steps:

1. **Gather your ingredients and tools**:
   - Teabags or loose-leaf tea
   - Hot water (about 200°F / 93°C for black tea, 185°F / 85°C for green tea)
   - A teapot or mug
   - Tea str
(8.3s)

Q: What is the capital of France?
A: The capital of France is Paris. It is located in the northern part of the country and is known for its iconic landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. Paris is also a major cultural, economic, and political 

In [5]:
import torch

  # keep only recent turns so prompt doesn't explode
history = []
tokenizer.truncation_side = "left"

def build_prompt(user_text, max_turns=6):
    turns = history[-max_turns:]
    prompt = ""
    for u, a in turns:
        prompt += f"Human: {u}\n\nAssistant: {a}\n\n"
    prompt += f"Human: {user_text}\n\nAssistant:"
    return prompt

def chat(user_text, max_new_tokens=140):
    prompt = build_prompt(user_text)
    x = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        y = model.generate(
            **x,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    ans = tokenizer.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    history.append((user_text, ans))
    print("Assistant:", ans)
    return ans

In [7]:
q = input("You: ")
chat(q)

You:  what is the speed of light?


Assistant: The speed of light in a vacuum is approximately **299,792 kilometers per second** (km/s) or about **1,079,252,848.8 miles per hour** (mph). This value is often rounded to 300,000 km/s for most practical purposes.

The exact value defined by the International System of Units (SI) is:

\[ c = 299,792,458 \text{ meters per second} \]

This constant plays a fundamental role in physics and is used extensively in various scientific calculations and theories, including Einstein's theory of relativity.


"The speed of light in a vacuum is approximately **299,792 kilometers per second** (km/s) or about **1,079,252,848.8 miles per hour** (mph). This value is often rounded to 300,000 km/s for most practical purposes.\n\nThe exact value defined by the International System of Units (SI) is:\n\n\\[ c = 299,792,458 \\text{ meters per second} \\]\n\nThis constant plays a fundamental role in physics and is used extensively in various scientific calculations and theories, including Einstein's theory of relativity."